In [10]:
import pandas as pd
import pickle
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [11]:
df_raw = pd.read_csv("DATASET - Sheet1.csv")


In [12]:
encoders = pickle.load(open("encoders.pkl","rb"))

In [13]:
df_encoded = df_raw.copy()
for col, le in encoders.items():
    if col not in df_encoded.columns:
        raise KeyError(f"Missing column in dataset: {col}")

    # Safe on re-run: skip columns already numerically encoded
    if pd.api.types.is_numeric_dtype(df_encoded[col]):
        continue

    values = df_encoded[col].astype(str).str.strip().str.upper()
    unknown = sorted(set(values.unique()) - set(le.classes_))
    if unknown:
        raise ValueError(f"{col} has unseen labels, e.g. {unknown[:5]}")

    df_encoded[col] = le.transform(values)


In [14]:
X_encoded = df_encoded.drop('WATER REQUIREMENT', axis=1)
X_raw = df_raw.drop('WATER REQUIREMENT', axis=1)
y = pd.to_numeric(df_raw['WATER REQUIREMENT'], errors='coerce')

valid_mask = y.notna()
X_encoded = X_encoded.loc[valid_mask]
X_raw = X_raw.loc[valid_mask]
y = y.loc[valid_mask]


In [15]:
rf = pickle.load(open("rf_model.pkl","rb"))
lr = pickle.load(open("lr_model.pkl","rb"))
xgb = pickle.load(open("xgb_model.pkl","rb"))


In [16]:
models = {
    "Random Forest": (rf, "encoded"),
    "Linear Regression": (lr, "raw"),
    "XGBoost": (xgb, "encoded")
}


In [19]:
for name, (model, input_type) in models.items():
    X_input = X_encoded if input_type == "encoded" else X_raw
    y_pred = model.predict(X_input)

    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)

    print(f"\n{name}")
    print(f"MAE: {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R2 Score: {r2:.3f}")



Random Forest
MAE: 0.986
RMSE: 8.785
R2 Score: 0.850

Linear Regression
MAE: 3.982
RMSE: 22.177
R2 Score: 0.044

XGBoost
MAE: 5.330
RMSE: 12.190
R2 Score: 0.711
